# Exporting the model to ONNX format

In [ ]:
import sys
sys.path.append('..')
from src.model import BertClassifier
from src.dataset import CuadDataset
from src.trainer import BertClassifierTrainer
import torch

In [ ]:
clause_bert = torch.load('../models/clause_bert_version_2_best.pt', map_location=torch.device('cpu'), weights_only=False)

In [ ]:
import torch

class BertONNXWrapper(torch.nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.model = original_model

    def forward(self, input_ids, attention_mask):
        encoding = {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }
        return self.model(encoding)

clause_bert.eval() 
model_for_export = BertONNXWrapper(clause_bert)
model_for_export.eval()

batch_size = 1
seq_length = 512

dummy_input_ids = torch.randint(0, 30522, (batch_size, seq_length), dtype=torch.long)
dummy_attention_mask = torch.ones((batch_size, seq_length), dtype=torch.long)

example_inputs = (dummy_input_ids, dummy_attention_mask)

torch.onnx.export(
    model_for_export,
    example_inputs,
    "../models/clause_bert.onnx",
    export_params=True,
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    
)

In [ ]:
import onnx
onnx_model = onnx.load('../models/clause_bert.onnx')